# Assignment 3 – Model Ensembling on Road Accident Risk
### Kaggle Competition: Playground Series S5E10 – Predicting Road Accident Risk
---
**Name:** Ashish Mulani &nbsp;&nbsp;|&nbsp;&nbsp; **PRN:** 260240128007  
**Name:** Pranita Lad &nbsp;&nbsp;|&nbsp;&nbsp; **PRN:** 260240128028

---
### Ensembling Techniques Implemented
| # | Technique | Description |
|---|-----------|-------------|
| 1 | **Bagging** | BaggingRegressor with GridSearchCV over base estimators & n_estimators |
| 2 | **Stacking** | StackingRegressor with multiple base learners + meta-learner tuned via GridSearchCV |
| 3 | **Voting / Averaging** | Manual soft-voting (average of predictions from top models) |
| 4 | **Boosting (GBM)** | GradientBoostingRegressor + XGBRegressor tuned via GridSearchCV |

> **Note:** No `train_test_split` is used anywhere. All evaluation is done via `KFold` cross-validation inside `GridSearchCV`.

## 1. Imports & Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, GridSearchCV, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error

# Base estimators
from sklearn.linear_model import LinearRegression, ElasticNet, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    BaggingRegressor,
    StackingRegressor,
    GradientBoostingRegressor,
    VotingRegressor
)
from xgboost import XGBRegressor

import os
# ── Change this path to your local Kaggle dataset folder ──
os.chdir("/kaggle/input/playground-series-s5e10")   # adjust if running locally

print("All imports successful ✓")

## 2. Load Data

In [ ]:
train = pd.read_csv('train.csv', index_col=0)
test  = pd.read_csv('test.csv',  index_col=0)
sample_sub = pd.read_csv('sample_submission.csv')

print(f"Train shape : {train.shape}")
print(f"Test  shape : {test.shape}")
train.head()

## 3. Exploratory Data Analysis

In [ ]:
print("=== Data Types ===")
print(train.dtypes)
print("\n=== Missing Values ===")
print(train.isnull().sum())
print("\n=== Target Summary ===")
print(train['accident_risk'].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Target distribution
axes[0].hist(train['accident_risk'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Target: accident_risk distribution')
axes[0].set_xlabel('accident_risk')
axes[0].set_ylabel('Frequency')

# Correlation heatmap (numeric only)
num_cols = train.select_dtypes(include=np.number).columns
corr = train[num_cols].corr()
sns.heatmap(corr, ax=axes[1], cmap='coolwarm', fmt='.2f', annot=len(num_cols) <= 12)
axes[1].set_title('Correlation Heatmap (numeric features)')

plt.tight_layout()
plt.show()

## 4. Preprocessing Pipeline

- **Categorical columns** → `OneHotEncoder` (drop first to avoid multicollinearity)
- **Numeric columns** → passed through as-is (`remainder='passthrough'`)
- **No train_test_split** — all scoring is via `GridSearchCV` with `KFold(n_splits=5)`

In [ ]:
X = train.drop('accident_risk', axis=1)
y = train['accident_risk']

# Cross-validation strategy (used inside ALL GridSearchCV calls)
kfold = KFold(n_splits=5, shuffle=True, random_state=26)

# Preprocessing transformer
ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore').set_output(transform='pandas')
preprocessor = ColumnTransformer(
    transformers=[('OHE', ohe, make_column_selector(dtype_include=object))],
    remainder='passthrough',
    verbose_feature_names_out=False
).set_output(transform='pandas')

print("Preprocessor ready ✓")
print(f"Features : {X.shape[1]}  |  Samples : {X.shape[0]}")

---
## 5. Technique 1 – BAGGING

**Concept:** Trains multiple instances of a base estimator on different bootstrap samples of the training data and averages their predictions.  
**GridSearchCV** tunes:
- `estimator` (base learner)
- `n_estimators` (number of bootstrap models)

In [ ]:
# Base estimators to try inside Bagging
lr  = LinearRegression()
el  = ElasticNet(random_state=26)
knn = KNeighborsRegressor()
dtr = DecisionTreeRegressor(random_state=26)

bagg = BaggingRegressor(random_state=26, n_jobs=-1)

pipe_bagg = Pipeline([
    ('PRE',  preprocessor),
    ('BAGG', bagg)
])

params_bagg = {
    'BAGG__estimator'    : [lr, el, knn, dtr],
    'BAGG__n_estimators' : [10, 25, 50, 100]
}

gcv_bagg = GridSearchCV(
    pipe_bagg, param_grid=params_bagg,
    cv=kfold, scoring='r2', n_jobs=-1, verbose=1
)
gcv_bagg.fit(X, y)

print(f"\nBest CV R² (Bagging) : {gcv_bagg.best_score_:.4f}")
print(f"Best Params          : {gcv_bagg.best_params_}")

In [ ]:
# Show full GridSearch results for Bagging
df_bagg = pd.DataFrame(gcv_bagg.cv_results_)
df_bagg['estimator_name'] = df_bagg['param_BAGG__estimator'].apply(
    lambda e: type(e).__name__
)
display(
    df_bagg[['estimator_name', 'param_BAGG__n_estimators', 'mean_test_score', 'std_test_score']]
    .sort_values('mean_test_score', ascending=False)
    .reset_index(drop=True)
)

In [ ]:
# --- BAGGING Submission ---
y_pred_bagg = gcv_bagg.best_estimator_.predict(test)
y_pred_bagg = np.clip(y_pred_bagg, 0, None)   # accident_risk cannot be negative

sub_bagg = sample_sub.copy()
sub_bagg['accident_risk'] = y_pred_bagg
sub_bagg.to_csv('submission_bagging.csv', index=False)
print("submission_bagging.csv saved ✓")
sub_bagg.head()

---
## 6. Technique 2 – STACKING

**Concept:** Level-0 models make predictions on out-of-fold data; those predictions become features for a Level-1 meta-learner.  
**GridSearchCV** tunes the meta-learner (`final_estimator`).

In [ ]:
# Level-0 base learners
base_learners = [
    ('LR',  LinearRegression()),
    ('EL',  ElasticNet(random_state=26)),
    ('KNN', KNeighborsRegressor(n_neighbors=7)),
    ('DTR', DecisionTreeRegressor(random_state=26, max_depth=5)),
    ('RF',  RandomForestRegressor(n_estimators=50, random_state=26, n_jobs=-1))
]

stack = StackingRegressor(
    estimators=base_learners,
    passthrough=True,           # pass original features to meta-learner too
    cv=5,
    n_jobs=-1
)

pipe_stack = Pipeline([
    ('PRE',   preprocessor),
    ('STACK', stack)
])

params_stack = {
    'STACK__final_estimator': [
        Ridge(alpha=1.0),
        Ridge(alpha=10.0),
        XGBRegressor(n_estimators=100, random_state=26, verbosity=0)
    ]
}

gcv_stack = GridSearchCV(
    pipe_stack, param_grid=params_stack,
    cv=kfold, scoring='r2', n_jobs=-1, verbose=1
)
gcv_stack.fit(X, y)

print(f"\nBest CV R² (Stacking) : {gcv_stack.best_score_:.4f}")
print(f"Best Meta-Learner     : {type(gcv_stack.best_params_['STACK__final_estimator']).__name__}")

In [ ]:
df_stack = pd.DataFrame(gcv_stack.cv_results_)
df_stack['meta_learner'] = df_stack['param_STACK__final_estimator'].apply(
    lambda e: type(e).__name__
)
display(
    df_stack[['meta_learner', 'mean_test_score', 'std_test_score']]
    .sort_values('mean_test_score', ascending=False)
    .reset_index(drop=True)
)

In [ ]:
# --- STACKING Submission ---
y_pred_stack = gcv_stack.best_estimator_.predict(test)
y_pred_stack = np.clip(y_pred_stack, 0, None)

sub_stack = sample_sub.copy()
sub_stack['accident_risk'] = y_pred_stack
sub_stack.to_csv('submission_stacking.csv', index=False)
print("submission_stacking.csv saved ✓")
sub_stack.head()

---
## 7. Technique 3 – BOOSTING (Gradient Boosting + XGBoost)

**Concept:** Sequentially trains weak learners where each new tree corrects the errors of the previous ensemble.  
**GridSearchCV** tunes `n_estimators`, `learning_rate`, and `max_depth` for both GBM and XGBoost, then we pick the best.

In [ ]:
# --- 7a. Gradient Boosting ---
gbm = GradientBoostingRegressor(random_state=26)
pipe_gbm = Pipeline([('PRE', preprocessor), ('GBM', gbm)])

params_gbm = {
    'GBM__n_estimators'  : [50, 100, 200],
    'GBM__learning_rate' : [0.05, 0.1, 0.2],
    'GBM__max_depth'     : [3, 4, 5]
}

gcv_gbm = GridSearchCV(
    pipe_gbm, param_grid=params_gbm,
    cv=kfold, scoring='r2', n_jobs=-1, verbose=1
)
gcv_gbm.fit(X, y)

print(f"\nBest CV R² (GBM)  : {gcv_gbm.best_score_:.4f}")
print(f"Best Params       : {gcv_gbm.best_params_}")

In [ ]:
# --- 7b. XGBoost ---
xgb = XGBRegressor(random_state=26, verbosity=0, n_jobs=-1)
pipe_xgb = Pipeline([('PRE', preprocessor), ('XGB', xgb)])

params_xgb = {
    'XGB__n_estimators'  : [50, 100, 200],
    'XGB__learning_rate' : [0.05, 0.1, 0.2],
    'XGB__max_depth'     : [3, 4, 6]
}

gcv_xgb = GridSearchCV(
    pipe_xgb, param_grid=params_xgb,
    cv=kfold, scoring='r2', n_jobs=-1, verbose=1
)
gcv_xgb.fit(X, y)

print(f"\nBest CV R² (XGBoost) : {gcv_xgb.best_score_:.4f}")
print(f"Best Params          : {gcv_xgb.best_params_}")

In [ ]:
# Pick the better boosting model
if gcv_gbm.best_score_ >= gcv_xgb.best_score_:
    best_boost = gcv_gbm
    boost_name = 'GradientBoosting'
else:
    best_boost = gcv_xgb
    boost_name = 'XGBoost'

print(f"Selected Boosting model: {boost_name}  (R²={best_boost.best_score_:.4f})")

# --- BOOSTING Submission ---
y_pred_boost = best_boost.best_estimator_.predict(test)
y_pred_boost = np.clip(y_pred_boost, 0, None)

sub_boost = sample_sub.copy()
sub_boost['accident_risk'] = y_pred_boost
sub_boost.to_csv('submission_boosting.csv', index=False)
print("submission_boosting.csv saved ✓")

---
## 8. Technique 4 – VOTING / AVERAGING ENSEMBLE

**Concept:** Combine predictions from the three best models (Bagging, Stacking, Boosting) by taking a simple average (soft voting). No extra hyperparameter tuning needed—just leverages diversity across the three different ensemble strategies.

This is equivalent to an equal-weight `VotingRegressor`.

In [ ]:
# Average predictions from all three best models
y_pred_avg = (y_pred_bagg + y_pred_stack + y_pred_boost) / 3.0
y_pred_avg = np.clip(y_pred_avg, 0, None)

sub_avg = sample_sub.copy()
sub_avg['accident_risk'] = y_pred_avg
sub_avg.to_csv('submission_voting_avg.csv', index=False)
print("submission_voting_avg.csv saved ✓")
sub_avg.head()

In [ ]:
# Weighted average — give more weight to the best CV score
scores_arr = np.array([
    gcv_bagg.best_score_,
    gcv_stack.best_score_,
    best_boost.best_score_
])
weights = scores_arr / scores_arr.sum()   # normalize to sum=1

y_pred_wavg = (
    weights[0] * y_pred_bagg +
    weights[1] * y_pred_stack +
    weights[2] * y_pred_boost
)
y_pred_wavg = np.clip(y_pred_wavg, 0, None)

sub_wavg = sample_sub.copy()
sub_wavg['accident_risk'] = y_pred_wavg
sub_wavg.to_csv('submission_weighted_avg.csv', index=False)
print("submission_weighted_avg.csv saved ✓")

print(f"\nWeights → Bagging: {weights[0]:.3f} | Stacking: {weights[1]:.3f} | Boosting: {weights[2]:.3f}")

---
## 9. Model Comparison

In [ ]:
comparison = pd.DataFrame({
    'Technique'    : ['Bagging', 'Stacking', f'Boosting ({boost_name})',
                      'Voting (Equal Avg)', 'Voting (Weighted Avg)'],
    'Best CV R²'   : [
        gcv_bagg.best_score_,
        gcv_stack.best_score_,
        best_boost.best_score_,
        '—  (derived)',
        '—  (derived)'
    ],
    'Submission File': [
        'submission_bagging.csv',
        'submission_stacking.csv',
        'submission_boosting.csv',
        'submission_voting_avg.csv',
        'submission_weighted_avg.csv'
    ]
})
display(comparison)

In [ ]:
# Bar chart of CV scores (only the three GridSearchCV ones)
cv_scores = {
    'Bagging' : gcv_bagg.best_score_,
    'Stacking': gcv_stack.best_score_,
    f'Boosting\n({boost_name})': best_boost.best_score_
}

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['steelblue', 'darkorange', 'seagreen']
bars = ax.bar(cv_scores.keys(), cv_scores.values(), color=colors, width=0.5)
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=11)
ax.set_ylim(0, 1)
ax.set_ylabel('5-Fold CV R² Score')
ax.set_title('Best Cross-Validation R² by Ensemble Technique')
ax.axhline(0, color='black', linewidth=0.7, linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of predictions across techniques
fig, ax = plt.subplots(figsize=(10, 4))
for preds, label, color in [
    (y_pred_bagg,  'Bagging',          'steelblue'),
    (y_pred_stack, 'Stacking',         'darkorange'),
    (y_pred_boost, f'Boosting ({boost_name})', 'seagreen'),
    (y_pred_avg,   'Equal Avg',        'red'),
    (y_pred_wavg,  'Weighted Avg',     'purple')
]:
    ax.hist(preds, bins=40, alpha=0.5, label=label, color=color)
ax.set_xlabel('Predicted accident_risk')
ax.set_ylabel('Count')
ax.set_title('Distribution of Predictions – All Ensemble Techniques')
ax.legend()
plt.tight_layout()
plt.show()

---
## 10. Summary & Kaggle Submission Instructions

### Files Generated
| File | Technique | Description |
|------|-----------|-------------|
| `submission_bagging.csv` | Bagging | BaggingRegressor (best base estimator & n_estimators via GridSearchCV) |
| `submission_stacking.csv` | Stacking | StackingRegressor (5 base learners + tuned meta-learner via GridSearchCV) |
| `submission_boosting.csv` | Boosting | Best of GBM / XGBoost (both tuned via GridSearchCV) |
| `submission_voting_avg.csv` | Voting – Equal | Simple average of all three above |
| `submission_weighted_avg.csv` | Voting – Weighted | CV-score-weighted average of all three above |

### How to Submit on Kaggle
1. Go to https://www.kaggle.com/competitions/playground-series-s5e10/submit
2. Upload each `.csv` file separately
3. Add a description (e.g., "Bagging – DTR base, n=100")
4. Compare Public LB scores across submissions

### Key Observations
- **GridSearchCV with KFold(n_splits=5)** was used for all model selection — no data leakage from train-test splitting.
- **Stacking** typically benefits most from diverse base learners; `passthrough=True` lets the meta-learner see raw features too.
- **Weighted averaging** tends to slightly outperform equal averaging by giving more weight to stronger models.
- Submit **all 5 files** and compare leaderboard scores to identify the strongest ensemble.